# Two gradient estimators, one 100× noisier

**Topic:** Reparameterization trick &nbsp;·&nbsp; **Difficulty:** Hard

You need the gradient of `E_{z ~ N(mu, sigma^2)}[f(z)]` with respect to `mu`. Two estimators are available. The score-function (REINFORCE) estimator
works for any distribution and is famously noisy. The [Policy
Gradients](/module/?id=policy-gradients) lesson teaches the same estimator
and the baseline its noise calls for. The reparameterization
estimator rewrites `z = mu + sigma * eps` and differentiates through the
sample. It only works for reparameterizable distributions, and it is
dramatically cheaper in variance.

Measure the gap. It is the reason VAEs train at all.

## The data (created in the next cell)

- `f(z) = (z - 0.7)^2 + 50` (note the constant). It changes nothing about the
  gradient being estimated, and everything about one of the estimators.
- `mu = 0.4`, `sigma = 1.2`.
- 2000 independent estimates, each from a minibatch of 32 samples, using
  `np.random.default_rng(500 + trial)`.

## Your task

With **NumPy only**:

1. Score-function estimator, per sample:
   `f(z) * d/dmu log q(z)` where `log q` is the Gaussian log-density, so
   `d/dmu log q = (z - mu) / sigma^2`. Average over the minibatch.
   Draw `z` the same way for both estimators: `eps = r.normal(0, 1, batch)` then
   `z = mu + sigma * eps`, with `r = np.random.default_rng(500 + trial)` per trial.
2. Reparameterization estimator: `z = mu + sigma * eps`, so
   `d/dmu f(z) = f'(z) * 1` with `f'(z) = 2(z - 0.7)`. Average over the minibatch.
3. The exact gradient is `d/dmu E[(z-0.7)^2] = 2(mu - 0.7)`.
4. Set `answers` to a dict with `"exact"`, `"mean_score"`, `"mean_reparam"`,
   `"std_score"`, `"std_reparam"`, `"variance_ratio"` =
   `std_score^2 / std_reparam^2`.
   `std_score` and `std_reparam` are the sample standard deviations of the
   2000 estimates, so compute them with `np.std(x, ddof=1)`. The reference
   values use `ddof=1`.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks both estimators land within three standard errors of the exact
gradient (the score-function one needs the wider window, which is itself
the result) and that the reparameterized estimator has at least 20x lower
variance.

In [ ]:
import numpy as np

mu, sigma = 0.4, 1.2
n_trials, batch = 2000, 32

def f(z):
    return (z - 0.7) ** 2 + 50.0

def df(z):
    return 2.0 * (z - 0.7)

print(f"mu = {mu}, sigma = {sigma}; exact gradient = {2 * (mu - 0.7):.4f}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("exact", [], -0.5999999999999999, 0.10046280888869948, 1e-06),
    ("mean_score", [], -0.5243391531452747, 0.08779430689216045, 1e-06),
    ("mean_reparam", [], -0.5960939921153174, 0.09980879468263847, 1e-06),
    ("std_score", [], 7.972395854234492, -1.3348821351483664, 1e-06),
    ("std_reparam", [], 0.4207230386026188, -0.07044503037034638, 1e-06),
    ("variance_ratio", [], 359.0749596922909, -60.122798420473494, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float).ravel()
    if list(_a.shape) not in ([], [1]):
        print(f"  {_key:<24} shape {list(_a.shape)}, expected a scalar or length-1 array")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "variance_ratio" in answers and "std_score" in answers and "std_reparam" in answers:
    _sem_s = answers["std_score"] / np.sqrt(n_trials)
    _sem_r = answers["std_reparam"] / np.sqrt(n_trials)
    _u1 = abs(answers["mean_score"] - answers["exact"]) < 3 * _sem_s
    _u2 = abs(answers["mean_reparam"] - answers["exact"]) < 3 * _sem_r
    _v = answers["variance_ratio"] > 20
    print(f"  {'both unbiased':<24} {'ok' if _u1 and _u2 else 'wrong'}  "
          f"(exact {answers['exact']:.4f}, score {answers['mean_score']:.4f}, "
          f"reparam {answers['mean_reparam']:.4f})")
    print(f"  {'reparam 20x lower var':<24} {'ok' if _v else 'wrong'}  "
          f"(ratio = {answers['variance_ratio']:.1f})")
    _ok = _ok and _u1 and _u2 and _v

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

Both estimators have the right expectation, so the difference is entirely
variance. The score-function estimator multiplies `f(z)` by a score term and never
looks at how `f` *changes*, only at its values, so the constant `+50` rides
straight into the variance while contributing nothing to the mean. The
reparameterized estimator differentiates `f`, and `d/dz (something + 50)` is just
`d/dz something`, so it does not notice the constant at all. This is exactly why
REINFORCE needs a baseline (subtract a constant to shrink the multiplier) and why
the reparameterization trick is what made VAEs trainable without one.

```python
import numpy as np

exact = 2.0 * (mu - 0.7)

score_est = np.zeros(n_trials)
reparam_est = np.zeros(n_trials)

for t in range(n_trials):
    r = np.random.default_rng(500 + t)
    eps = r.normal(0, 1, batch)
    z = mu + sigma * eps
    score_est[t] = float(np.mean(f(z) * (z - mu) / sigma ** 2))
    reparam_est[t] = float(np.mean(df(z)))

answers = {
    "exact": float(exact),
    "mean_score": float(score_est.mean()),
    "mean_reparam": float(reparam_est.mean()),
    "std_score": float(score_est.std(ddof=1)),
    "std_reparam": float(reparam_est.std(ddof=1)),
    "variance_ratio": float(score_est.var(ddof=1) / reparam_est.var(ddof=1)),
}
```